In [ ]:
import finnhub
import os
from dotenv import load_dotenv

load_dotenv()
finnhub_client = finnhub.Client(api_key=os.getenv("FINNHUB_API_KEY"))

data = finnhub_client.company_basic_financials(symbol="AAPL", metric="all")

In [8]:
# See what's inside annual vs quarterly
print("=== SERIES KEYS ===")
for k in data["series"].keys():
    print(f"  {k}")

print("\n=== ANNUAL KEYS ===")
for k in data["series"]["annual"].keys():
    print(f"  {k}")

print("\n=== QUARTERLY KEYS ===")
for k in data["series"]["quarterly"].keys():
    print(f"  {k}")

=== SERIES KEYS ===
  annual
  quarterly

=== ANNUAL KEYS ===
  bookValue
  cashRatio
  currentRatio
  ebitPerShare
  eps
  ev
  evEbitda
  evRevenue
  fcfMargin
  grossMargin
  inventoryTurnover
  longtermDebtTotalAsset
  longtermDebtTotalCapital
  longtermDebtTotalEquity
  netDebtToTotalCapital
  netDebtToTotalEquity
  netMargin
  operatingMargin
  payoutRatio
  pb
  pe
  pfcf
  pretaxMargin
  ps
  ptbv
  quickRatio
  receivablesTurnover
  roa
  roe
  roic
  rotc
  salesPerShare
  sgaToSale
  tangibleBookValue
  totalDebtToEquity
  totalDebtToTotalAsset
  totalDebtToTotalCapital
  totalRatio

=== QUARTERLY KEYS ===
  assetTurnoverTTM
  bookValue
  cashRatio
  currentRatio
  ebitPerShare
  eps
  ev
  evEbitdaTTM
  evRevenueTTM
  fcfMargin
  fcfPerShareTTM
  grossMargin
  inventoryTurnoverTTM
  longtermDebtTotalAsset
  longtermDebtTotalCapital
  longtermDebtTotalEquity
  netDebtToTotalCapital
  netDebtToTotalEquity
  netMargin
  operatingMargin
  payoutRatioTTM
  pb
  peTTM
  pfcfTTM
 

In [9]:
# Pick one metric from each to see the time-series shape
first_annual_key = next(iter(data["series"]["annual"]))
first_quarterly_key = next(iter(data["series"]["quarterly"]))

print(f"=== ANNUAL SAMPLE: {first_annual_key} ===")
for entry in data["series"]["annual"][first_annual_key][:3]:
    print(f"  {entry}")

print(f"\n=== QUARTERLY SAMPLE: {first_quarterly_key} ===")
for entry in data["series"]["quarterly"][first_quarterly_key][:3]:
    print(f"  {entry}")

=== ANNUAL SAMPLE: bookValue ===
  {'period': '2025-09-27', 'v': 73733}
  {'period': '2024-09-28', 'v': 56950}
  {'period': '2023-09-30', 'v': 62146}

=== QUARTERLY SAMPLE: assetTurnoverTTM ===
  {'period': '2025-12-27', 'v': 1.2435}
  {'period': '2025-09-27', 'v': 1.2186}
  {'period': '2025-06-28', 'v': 1.1915}


In [10]:
print(f"Total flat metrics:       {len(data['metric'])}")
print(f"Annual series metrics:    {len(data['series']['annual'])}")
print(f"Quarterly series metrics: {len(data['series']['quarterly'])}")

Total flat metrics:       132
Annual series metrics:    38
Quarterly series metrics: 40


### ─────────────────────────────────────────────
### SECTION 2 — PRODUCTION RUN (ALL 60 COMPANIES)
### ─────────────────────────────────────────────

In [11]:
import finnhub
import os
import time
from dotenv import load_dotenv

load_dotenv()
finnhub_client = finnhub.Client(api_key=os.getenv("FINNHUB_API_KEY"))

tickers = [
    # DEFENSE - High Lobby
    "LMT", "RTX", "NOC", "GD", "BA", "LHX", "LDOS", "HII", "BAESY", "SAIC",
    # DEFENSE - Low Lobby
    "TXT", "TDG", "HEI", "DRS", "KTOS", "AVAV", "MRCY", "CW", "MOG.A", "DCO",
    # ENERGY - High Lobby
    "XOM", "CVX", "COP", "OXY", "BP", "NEE", "D", "DUK", "HAL", "BKR",
    # ENERGY - Low Lobby
    "SLB", "VLO", "PSX", "EOG", "FANG", "DVN", "CTRA", "AR", "CHRD", "MTDR",
    # TECH - High Lobby
    "MSFT", "AMZN", "GOOGL", "IBM", "ORCL", "PLTR", "BAH", "CACI", "PSN", "CRM",
    # TECH - Low Lobby
    "AAPL", "META", "NVDA", "CSCO", "PANW", "CRWD", "SNOW", "DDOG", "NET", "TWLO"
]

# Duplicate guard
duplicates = [t for t in tickers if tickers.count(t) > 1]
assert not duplicates, f"Duplicate tickers found: {duplicates}"

results = {}
issues = []

for i, ticker in enumerate(tickers):
    try:
        data = finnhub_client.company_basic_financials(symbol=ticker, metric="all")
        results[ticker] = data

        if not data or not data.get("metric"):
            issues.append((ticker, "empty response — ticker may be unavailable"))
            continue

        none_fields = [k for k, v in data["metric"].items() if v is None or v == ""]
        if none_fields:
            issues.append((ticker, f"missing metric fields: {none_fields}"))

        if not data.get("series"):
            issues.append((ticker, "series data missing entirely"))
        else:
            if not data["series"].get("annual"):
                issues.append((ticker, "annual series missing"))
            if not data["series"].get("quarterly"):
                issues.append((ticker, "quarterly series missing"))

    except Exception as e:
        issues.append((ticker, f"API error: {str(e)}"))
        results[ticker] = {}

    if i < len(tickers) - 1:
        time.sleep(1)

# Summary
successful = [t for t, r in results.items() if r and r.get("metric")]
print(f"✅ Successfully pulled: {len(successful)} / {len(tickers)} companies")
print(f"⚠️  Issues found: {len(issues)}")
for ticker, issue in issues:
    print(f"   {ticker}: {issue}")

empty = [ticker for ticker, data in results.items() if not data or not data.get("metric")]
print(f"\nEmpty responses: {empty if empty else 'None'}")

✅ Successfully pulled: 60 / 60 companies
⚠️  Issues found: 37
   RTX: missing metric fields: ['epsGrowth5Y', 'netMarginGrowth5Y']
   BA: missing metric fields: ['ebitdaCagr5Y', 'ebitdaInterimCagr5Y', 'epsGrowth3Y', 'epsGrowth5Y', 'epsGrowthQuarterlyYoy', 'epsGrowthTTMYoy', 'focfCagr5Y', 'netMarginGrowth5Y']
   LDOS: missing metric fields: ['epsGrowthQuarterlyYoy']
   DRS: missing metric fields: ['focfCagr5Y']
   KTOS: missing metric fields: ['currentDividendYieldTTM', 'dividendPerShareTTM', 'epsGrowth3Y', 'focfCagr5Y', 'pcfShareAnnual', 'pcfShareTTM']
   AVAV: missing metric fields: ['currentDividendYieldTTM', 'dividendPerShareTTM', 'epsGrowth3Y', 'epsGrowthQuarterlyYoy', 'epsGrowthTTMYoy', 'focfCagr5Y', 'pcfShareAnnual', 'pcfShareTTM', 'peBasicExclExtraTTM', 'peExclExtraTTM', 'peInclExtraTTM', 'peTTM']
   MRCY: missing metric fields: ['currentDividendYieldTTM', 'dividendPerShareTTM', 'ebitdaInterimCagr5Y', 'epsGrowth3Y', 'epsGrowth5Y', 'epsGrowthQuarterlyYoy', 'epsGrowthTTMYoy', 'netM